# Project Overview

Dataset: https://www.kaggle.com/datasets/nathanlauga/nba-games?select=teams.csv

### AI References
chi square helper function

### EDA

In [1]:
import pandas as pd
import numpy as np
import networkx as nx

In [2]:
games = pd.read_csv("/Users/leowiemelt/Desktop/CSE150A/data/games.csv")
teams = pd.read_csv("/Users/leowiemelt/Desktop/CSE150A/data/teams.csv")
ranking = pd.read_csv("/Users/leowiemelt/Desktop/CSE150A/data/ranking.csv")

In [3]:
print(f"Home team win percentage: {games['HOME_TEAM_WINS'].mean()}")

Home team win percentage: 0.5870323815241454


In [4]:
# PointDiff: point differential (home - away), binned into large loss/win (+-10 points) and close games
# Will NOT be included as a predictor in the model, just to see causal effects of other vars
games["PointDiff"] = games["PTS_home"] - games["PTS_away"]

games["PointDiff_bin"] = pd.cut(
    games["PointDiff"],
    bins=[-np.inf, -10, 10, np.inf],
    labels=["LargeLoss", "Close", "LargeWin"]
)

### Data Preprocessing and Thresholding

Apply appropriate thresholding or discretization techniques to your variables as needed
for Bayesian Network modeling.
Clearly justify the thresholds or binning strategies you choose.
Reference insights from your prior EDA where relevant.

In [5]:
# Target variable: HOME_TEAM_WINS (bool)
games["HomeWin"] = games["HOME_TEAM_WINS"].astype(int)

In [8]:
# # Home and Away win percentage: bin into weak (<=40%), average (40-60%), and strong (>=60%)
# # Standard cutoffs commonly used in basketball analytics, reduces noise of using raw win %
# # WinPctDiff = HomeWinPct - AwayWinPct, binned into AwayStronger, Even, or HomeStronger for diff >= 10%

games["GAME_DATE_EST"] = pd.to_datetime(games["GAME_DATE_EST"])
ranking["STANDINGSDATE"] = pd.to_datetime(ranking["STANDINGSDATE"])


games = games.sort_values("GAME_DATE_EST")
ranking = ranking.sort_values("STANDINGSDATE")
home_games = games[["GAME_ID", "HOME_TEAM_ID", "GAME_DATE_EST"]].copy()

home_games = pd.merge_asof(
    home_games,
    ranking[["TEAM_ID", "STANDINGSDATE", "W_PCT"]],
    left_on="GAME_DATE_EST",
    right_on="STANDINGSDATE",
    left_by="HOME_TEAM_ID",
    right_by="TEAM_ID",
    direction="backward"
)

home_games.rename(columns={"W_PCT": "HomeWinPct"}, inplace=True)
home_games = home_games[["GAME_ID", "HomeWinPct"]]

away_games = games[["GAME_ID", "VISITOR_TEAM_ID", "GAME_DATE_EST"]].copy()

away_games = pd.merge_asof(
    away_games,
    ranking[["TEAM_ID", "STANDINGSDATE", "W_PCT"]],
    left_on="GAME_DATE_EST",
    right_on="STANDINGSDATE",
    left_by="VISITOR_TEAM_ID",
    right_by="TEAM_ID",
    direction="backward"
)

away_games.rename(columns={"W_PCT": "AwayWinPct"}, inplace=True)
away_games = away_games[["GAME_ID", "AwayWinPct"]]
games = games.merge(home_games, on="GAME_ID", how="left")
games = games.merge(away_games, on="GAME_ID", how="left")


In [ ]:
games.describe()

,GAME_DATE_EST,GAME_ID,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,FG3_PCT_home,...,PointDiff,HomeWin,HomeWinPct_x,AwayWinPct_x,WinPctDiff,HomeRestDays,AwayRestDays,RestDiff,HomeWinPct_y,AwayWinPct_y
count,27301,2.730100e+04,2.730100e+04,2.730100e+04,27301.000000,2.730100e+04,27202.000000,27202.000000,27202.000000,27202.000000,...,27202.000000,27301.000000,27301.000000,27301.000000,27301.000000,27301.000000,27301.000000,27301.000000,27301.000000,27301.000000
mean,2013-05-12 02:12:45.598329856,2.176070e+07,1.610613e+09,1.610613e+09,2012.301637,1.610613e+09,103.703514,0.461178,0.759455,0.356221,...,2.710867,0.587561,0.509387,0.500240,0.009147,3.976374,3.752134,0.224241,0.509387,0.500240
min,2003-10-05 00:00:00,1.030000e+07,1.610613e+09,1.610613e+09,2003.000000,1.610613e+09,36.000000,0.250000,0.143000,0.000000,...,-68.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,-191.000000,0.000000,0.000000
25%,2008-04-04 00:00:00,2.070017e+07,1.610613e+09,1.610613e+09,2007.000000,1.610613e+09,95.000000,0.422000,0.696000,0.286000,...,-7.000000,0.000000,0.379000,0.375000,-0.167000,2.000000,1.000000,0.000000,0.379000,0.375000
50%,2013-04-09 00:00:00,2.120109e+07,1.610613e+09,1.610613e+09,2012.000000,1.610613e+09,103.000000,0.461000,0.765000,0.356000,...,4.000000,1.000000,0.510000,0.500000,0.000000,2.000000,2.000000,0.000000,0.510000,0.500000
75%,2018-03-03 00:00:00,2.180053e+07,1.610613e+09,1.610613e+09,2017.000000,1.610613e+09,112.000000,0.500000,0.829000,0.429000,...,11.000000,1.000000,0.642000,0.634000,0.180000,3.000000,2.000000,1.000000,0.642000,0.634000
max,2022-12-22 00:00:00,5.210021e+07,1.610613e+09,1.610613e+09,2022.000000,1.610613e+09,168.000000,0.684000,1.000000,1.000000,...,73.000000,1.000000,1.000000,1.000000,1.000000,277.000000,275.000000,200.000000,1.000000,1.000000
std,NaN,5.503605e+06,8.618097e+00,8.647076e+00,5.649529,8.618097e+00,13.370318,0.056645,0.101018,0.111026,...,13.688549,0.492282,0.200920,0.199903,0.309160,16.888158,16.637080,11.684888,0.200920,0.199903


In [ ]:
# Win percentage must be between 0 and 1
assert games["HomeWinPct"].between(0, 1).all()
assert games["AwayWinPct"].between(0, 1).all()

# Stronger team should win more often (aggregate sanity check)
games.groupby("WinPctDiff_bin")["HomeWin"].mean()


KeyError: 'HomeWinPct'

In [ ]:
def win_pct_bin(p):
    if p <= 0.40:
        return "Weak"
    elif p <= 0.60:
        return "Average"
    else:
        return "Strong"

games["HomeStrength"] = games["HomeWinPct"].apply(win_pct_bin)
games["AwayStrength"] = games["AwayWinPct"].apply(win_pct_bin)

games["WinPctDiff"] = games["HomeWinPct"] - games["AwayWinPct"]

games["WinPctDiff_bin"] = pd.cut(
    games["WinPctDiff"],
    bins=[-np.inf, -0.10, 0.10, np.inf],
    labels=["AwayStronger", "Even", "HomeStronger"]
)


In [ ]:
def win_pct_bin(x):
    if x < 0.40:
        return "Weak"
    elif x <= 0.60:
        return "Average"
    else:
        return "Strong"

games["HomeWinPct_bin"] = games["HomeWinPct"].apply(win_pct_bin)
games["AwayWinPct_bin"] = games["AwayWinPct"].apply(win_pct_bin)

In [ ]:
# HomeRestDays, AwayRestDays: BackToBack = 0 days, Normal = 1-2 days, Rested = 3 days or more
# RestDiff = HomeRestDays - AwayRestDays. Binned into Home, Away, or no rest advantage
games = games.sort_values("GAME_DATE_EST")

last_game = {}
home_rest = []
away_rest = []

for _, row in games.iterrows():
    date = row["GAME_DATE_EST"]
    home = row["HOME_TEAM_ID"]
    away = row["VISITOR_TEAM_ID"]

    home_rest.append(
        (date - last_game.get(home, date)).days if home in last_game else 2
    )
    away_rest.append(
        (date - last_game.get(away, date)).days if away in last_game else 2
    )

    last_game[home] = date
    last_game[away] = date

games["HomeRestDays"] = home_rest
games["AwayRestDays"] = away_rest

In [ ]:
def rest_bin(days):
    if days == 0:
        return "BackToBack"
    elif days <= 2:
        return "NormalRest"
    else:
        return "WellRested"

games["HomeRest_bin"] = games["HomeRestDays"].apply(rest_bin)
games["AwayRest_bin"] = games["AwayRestDays"].apply(rest_bin)

In [ ]:
games["RestDiff"] = games["HomeRestDays"] - games["AwayRestDays"]
games["RestDiff_bin"] = pd.cut(games["RestDiff"], bins=[-np.inf, -1, 0, np.inf], labels=["AwayRestAdv", "EvenRest", "HomeRestAdv"])

In [ ]:
data = games[["HOME_TEAM_WINS", "HomeStrength", "AwayStrength", "WinPctDiff_bin", "HomeRest_bin", "AwayRest_bin", "RestDiff_bin"]].dropna()

## Conditional Independence Testing

In [ ]:
from scipy.stats import chi2_contingency

# tests if X and Y are CI given Z using chi-square test
def conditional_chi_square(data, X, Y, Z):
    p_values = []

    for z_val in data[Z].unique():
        subset = data[data[Z] == z_val]
        if subset[X].nunique() < 2 or subset[Y].nunique() < 2:
            continue

        contingency = pd.crosstab(subset[X], subset[Y])
        _, p, _, _ = chi2_contingency(contingency)
        p_values.append(p)

    return max(p_values) if p_values else 1.0


In [ ]:
conditional_chi_square(data, X="HomeStrength", Y="HOME_TEAM_WINS", Z="WinPctDiff_bin")

np.float64(5.758673089028558e-12)

In [ ]:
games.iloc[500:510, 20:40]

,HOME_TEAM_WINS,PointDiff,PointDiff_bin,HomeWin,HomeWinPct,HomeRecord,AwayWinPct,AwayRecord,HomeStrength,AwayStrength,WinPctDiff,WinPctDiff_bin,HomeWinPct_bin,AwayWinPct_bin,HomeRestDays,AwayRestDays,HomeRest_bin,AwayRest_bin,RestDiff,RestDiff_bin
18809,1,22.0,LargeWin,1,0.500,8-7,0.241,5-10,Average,Weak,0.259,HomeStronger,Average,Weak,1,1,NormalRest,NormalRest,0,EvenRest
18808,1,7.0,Close,1,0.607,11-3,0.556,6-9,Strong,Average,0.051,Even,Strong,Average,2,2,NormalRest,NormalRest,0,EvenRest
18798,1,11.0,LargeWin,1,0.321,7-6,0.621,9-7,Weak,Strong,-0.300,AwayStronger,Weak,Strong,3,3,WellRested,WellRested,0,EvenRest
18797,1,5.0,Close,1,0.267,5-9,0.483,6-7,Weak,Average,-0.216,AwayStronger,Weak,Average,3,2,WellRested,NormalRest,1,HomeRestAdv
18799,1,7.0,Close,1,0.393,9-4,0.308,4-12,Weak,Weak,0.085,Even,Weak,Weak,2,4,NormalRest,WellRested,-2,AwayRestAdv
18795,1,21.0,LargeWin,1,0.667,12-3,0.458,3-8,Strong,Average,0.209,HomeStronger,Strong,Average,3,3,WellRested,WellRested,0,EvenRest
18796,0,-6.0,Close,0,0.345,7-7,0.667,11-4,Weak,Strong,-0.322,AwayStronger,Weak,Strong,3,3,WellRested,WellRested,0,EvenRest
18801,1,9.0,Close,1,0.500,13-4,0.556,3-10,Average,Average,-0.056,Even,Average,Average,2,3,NormalRest,WellRested,-1,AwayRestAdv
18800,0,-8.0,Close,0,0.556,9-3,0.724,10-5,Average,Strong,-0.168,AwayStronger,Average,Strong,3,3,WellRested,WellRested,0,EvenRest
18803,1,17.0,LargeWin,1,0.769,15-1,0.536,6-10,Strong,Average,0.233,HomeStronger,Strong,Average,2,1,NormalRest,NormalRest,1,HomeRestAdv
